In [ ]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import HumanMessage

load_dotenv()

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server" : {
            "transport" : "stremable_http",
            "url" : "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

travel_agent = create_agent("google_genai:gemini-2.5-flash",
                     tools=tools,
                     checkpointer=InMemorySaver(),
                     system_prompt="You are a travel agent. Your job is to know the users needs and recommend flights according to that and ask the follow up questions if required for the flight"
                     )


In [ ]:
@tool
def travel_specialist(query: str) -> str:
    """Delegate flight, pricing, and travel search requests to the travel specialist"""

    result = travel_agent.invoke(
        {"messages" : [HumanMessage(content=query)]},
        {"configurable" : {"thread_id" : "isolated_travel_thread"}}
    )

    return result["messages"][-1].content
